# Test RAGAS

In [44]:
import sys
sys.path.insert(0, '/home/local/QCRI/fdeniz/projects/sspbench')

from sspbench.novelty.llm_utils import create_model_from_config
from sspbench.novelty.ragas_utils import (
    is_ragas_available,
    generate_qa_with_ragas,
    evaluate_qa_faithfulness,
    SentenceTransformerEmbeddings
)

print("✓ Imports successful")
print(f"RAGAS available: {is_ragas_available()}")

✓ Imports successful
RAGAS available: True


## Setup Eval Model

In [45]:
import os
# Set dummy OpenAI API key to prevent RAGAS from requiring real OpenAI credentials
os.environ["OPENAI_API_KEY"] = "dummy-key-for-ragas"

eval_config = {
    "type": "openai",
    "model": "gpt-oss",
    "api_url": "http://10.4.8.217:8000/v1",
    "api_token": "abc123",
    "api_version": "2024-12-01-preview"
}

try:
    eval_model = create_model_from_config(eval_config)
    print(f"✓ eval_model created successfully: {type(eval_model)}. Sample response: {eval_model.generate('Hello')}")
except Exception as e:
    print(f"✗ Failed to create eval_model: {e}")
    eval_model = None

embedding_model = SentenceTransformerEmbeddings("all-MiniLM-L6-v2")

Using cached model for config: gpt-oss
Generating with messages: [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'Hello'}]


gpt-oss: Hello! How can I assist you today?
✓ eval_model created successfully: <class 'models.openai_model.OpenaiLLM'>. Sample response: ['Hello! How can I assist you today?']


In [46]:
# Sample paragraph for testing
test_paragraph = """
The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. 
It is named after the engineer Gustave Eiffel, whose company designed and built the tower. 
Constructed from 1887 to 1889 as the entrance arch to the 1889 World's Fair, it was initially 
criticized by some of France's leading artists and intellectuals for its design, but it has 
become a global cultural icon of France and one of the most recognizable structures in the world.
""".strip()

print("Test paragraph:")
print(test_paragraph)
print("\n" + "="*80 + "\n")

Test paragraph:
The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. 
It is named after the engineer Gustave Eiffel, whose company designed and built the tower. 
Constructed from 1887 to 1889 as the entrance arch to the 1889 World's Fair, it was initially 
criticized by some of France's leading artists and intellectuals for its design, but it has 
become a global cultural icon of France and one of the most recognizable structures in the world.




In [47]:
# Configure RAGAS synthesizers for shorter answers
from ragas.testset.synthesizers import SingleHopSpecificQuerySynthesizer
from ragas.testset import TestsetGenerator
from sspbench.novelty.ragas_utils import CustomRagasLLM, CustomRagasEmbeddings

# Create RAGAS adapters
ragas_llm = CustomRagasLLM(eval_model, temperature=0.0)
ragas_embeddings = CustomRagasEmbeddings(embedding_model)

# Configure query distribution for ONLY short, specific questions
query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=ragas_llm), 1.0),  # 100% single-hop specific for shortest answers
]

print("✓ Configured query distribution for short answers:")
for synthesizer, weight in query_distribution:
    print(f"  {synthesizer.__class__.__name__}: {weight}")
print()

✓ Configured query distribution for short answers:
  SingleHopSpecificQuerySynthesizer: 1.0



In [48]:
if eval_model and is_ragas_available():
    print("Generating Q&A pairs with RAGAS using synthesizers...\n")
    try:
        # Create generator with synthesizer configuration
        generator = TestsetGenerator(
            llm=ragas_llm,
            embedding_model=ragas_embeddings,
        )

        from langchain_core.documents import Document
        doc = Document(page_content=test_paragraph)

        query_distribution = [
            (SingleHopSpecificQuerySynthesizer(llm=ragas_llm), 1.0),
        ]
        testset = generator.generate_with_langchain_docs(
            documents=[doc],
            testset_size=3,
            query_distribution=query_distribution
        )

        qa_pairs = []
        for idx, sample in enumerate(testset.samples, 1):
            qa_pairs.append(
                {
                    "id": str(idx),
                    "question": sample.eval_sample.user_input,
                    "answer": sample.eval_sample.reference,
                    "difficulty": "2",
                }
            )

        print(f"✓ Generated {len(qa_pairs)} Q&A pairs:\n")
        for i, qa in enumerate(qa_pairs, 1):
            print(f"Q{i}: {qa['question']}")
            print(f"A{i}: {qa['answer']}")
            print(f"Difficulty: {qa['difficulty']}")
            print("-" * 80)
    except Exception as e:
        print(f"✗ Error generating Q&A: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️ Skipping test - eval_model or RAGAS not available")

Generating Q&A pairs with RAGAS using synthesizers...



Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Generating with messages: [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'Summarize the given text in less than 10 sentences.\nPlease return the output in a JSON format that complies with the following schema as specified in JSON Schema:\n{"properties": {"text": {"title": "Text", "type": "string"}}, "required": ["text"], "title": "StringIO", "type": "object"}Do not use single quotes in your response but double quotes,properly escaped with a backslash.\n\n--------EXAMPLES-----------\nExample 1\nInput: {\n    "text": "Artificial intelligence\\n\\nArtificial intelligence is transforming various industries by automating tasks that previously required human intelligence. From healthcare to finance, AI is being used to analyze vast amounts of data quickly and accurately. This technology is also driving innovations in areas like self-driving cars and personalized recommendations."\n}\nOutput: {\n    "text": "AI is revolutionizing industries by auto

Applying CustomNodeFilter:   0%|          | 0/1 [00:00<?, ?it/s]

Generating with messages: [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'Given a document summary and node content, score the content of the node in 1 to 5 range.\nPlease return the output in a JSON format that complies with the following schema as specified in JSON Schema:\n{"properties": {"score": {"description": "1 to 5 score", "title": "Score", "type": "integer"}}, "required": ["score"], "title": "QuestionPotentialOutput", "type": "object"}Do not use single quotes in your response but double quotes,properly escaped with a backslash.\n\n-----------------------------\n\nNow perform the same with the following input\ninput: {\n    "document_summary": "The Eiffel Tower, a wrought-iron lattice tower built for the 1889 World’s Fair and named after engineer Gustave Eiffel, was once criticized but has become a global cultural icon and one of the world’s most recognizable structures.",\n    "node_content": "The Eiffel Tower is a wrought-iron lat

...

Generating with messages: [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'Using the provided summary, generate a single persona who would likely interact with or benefit from the content. Include a unique name and a concise role description of who they are.\nPlease return the output in a JSON format that complies with the following schema as specified in JSON Schema:\n{"properties": {"name": {"title": "Name", "type": "string"}, "role_description": {"title": "Role Description", "type": "string"}}, "required": ["name", "role_description"], "title": "Persona", "type": "object"}Do not use single quotes in your response but double quotes,properly escaped with a backslash.\n\n--------EXAMPLES-----------\nExample 1\nInput: {\n    "text": "Guide to Digital Marketing explains strategies for engaging audiences across various online platforms."\n}\nOutput: {\n    "name": "Digital Marketing Specialist",\n    "role_description": "Focuses on engaging audi

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating with messages: [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'Given a list of themes and personas with their roles, associate each persona with relevant themes based on their role description.\nPlease return the output in a JSON format that complies with the following schema as specified in JSON Schema:\n{"properties": {"mapping": {"additionalProperties": {"items": {"type": "string"}, "type": "array"}, "title": "Mapping", "type": "object"}}, "required": ["mapping"], "title": "PersonaThemesMapping", "type": "object"}Do not use single quotes in your response but double quotes,properly escaped with a backslash.\n\n--------EXAMPLES-----------\nExample 1\nInput: {\n    "themes": [\n        "Empathy",\n        "Inclusivity",\n        "Remote work"\n    ],\n    "personas": [\n        {\n            "name": "HR Manager",\n            "role_description": "Focuses on inclusivity and employee support."\n        },\n        {\n            "n

Generating Samples:   0%|          | 0/3 [00:00<?, ?it/s]

Generating with messages: [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'Generate a single-hop query and answer based on the specified conditions (persona, term, style, length) and the provided context. Ensure the answer is entirely faithful to the context, using only the information directly from the provided context.### Instructions:\n1. **Generate a Query**: Based on the context, persona, term, style, and length, create a question that aligns with the persona\'s perspective and incorporates the term.\n2. **Generate an Answer**: Using only the content from the provided context, construct a detailed answer to the query. Do not add any information not included in or inferable from the context.\n3. **Additional Context** (if provided): If llm_context is provided, use it as guidance for what type of question to generate (e.g., comparison questions, how-to questions, application-based questions) and how to structure the answer accordingly. Sti

## Generate Q&A Pairs with Faithfulness Evaluation

In [51]:
# Evaluate each Q&A pair for faithfulness and relevancy
if 'qa_pairs' in locals() and qa_pairs:
    print("Evaluating Q&A pairs for faithfulness and relevancy...\n")
    for i, qa in enumerate(qa_pairs, 1):
        question = qa['question']
        answer = qa['answer']
        
        print(f"Evaluating Q&A pair {i}:")
        print(f"Question: {question}")
        print(f"Answer: {answer}")
        
        # Simple faithfulness evaluation using our custom model
        try:
            # Create a simple prompt to evaluate faithfulness
            eval_prompt = f"""
Given the question: "{question}"
And the answer: "{answer}"
And the context: "{test_paragraph}"

Rate the faithfulness of the answer on a scale of 0-1, where:
- 1.0 = The answer is completely faithful to the context
- 0.0 = The answer contradicts or is not supported by the context

Also rate the answer relevancy on a scale of 0-1, where:
- 1.0 = The answer directly and completely answers the question
- 0.0 = The answer does not address the question

Respond with just two numbers separated by a space, like: 0.95 0.88
"""

            eval_response = eval_model.generate(eval_prompt)
            # Parse the response
            parts = eval_response[0].strip().split()
            if len(parts) >= 2:
                faithfulness_score = float(parts[0])
                relevancy_score = float(parts[1])
            else:
                # Fallback if parsing fails
                faithfulness_score = 0.5
                relevancy_score = 0.5

            print(f"Faithfulness: {faithfulness_score:.4f}")
            print(f"Answer Relevancy: {relevancy_score:.4f}")

        except Exception as e:
            print(f"Error in custom evaluation: {e}")
            print("Using fallback scores: 0.5")
            print(f"Faithfulness: 0.50")
            print(f"Answer Relevancy: 0.50")
        
        print("-" * 80)
else:
    print("⚠️ No qa_pairs found. Please run the Q&A generation cell first.")

Evaluating Q&A pairs for faithfulness and relevancy...

Evaluating Q&A pair 1:
Question: Where on the Champ de Mars is the Eiffel Tower situated?
Answer: The Eiffel Tower is a wrought‑iron lattice tower on the Champ de Mars in Paris, France.
Generating with messages: [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': '\nGiven the question: "Where on the Champ de Mars is the Eiffel Tower situated?"\nAnd the answer: "The Eiffel Tower is a wrought‑iron lattice tower on the Champ de Mars in Paris, France."\nAnd the context: "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. \nIt is named after the engineer Gustave Eiffel, whose company designed and built the tower. \nConstructed from 1887 to 1889 as the entrance arch to the 1889 World\'s Fair, it was initially \ncriticized by some of France\'s leading artists and intellectuals for its design, but it has \nbecome a global cultural icon of France and one of the mo

## Summary

This notebook demonstrates:
1. ✓ RAGAS setup with SingleHopSpecificQuerySynthesizer
2. ✓ Q&A pair generation with faithfulness evaluation for each pair
3. ✓ Clean, focused testing of the RAGAS integration

Each generated Q&A pair includes faithfulness and answer relevancy scores.